# Global Shipping Chokepoints — 05: A Real Map, Rendered in Databricks

The hand-drawn SVG map used for the web version doesn't read as a "real" map --
no coastline detail, no place labels, no basemap. This notebook renders the same real
data with an actual basemap (streets/labels via free, tokenless map tiles) using Plotly's
newer MapLibre-based traces (`Scattermap` / `Densitymap`, not the older `*mapbox`
versions) -- no Mapbox account, no card, no token needed for any of this. `display()`
in Databricks renders the result natively. Take a screenshot of whichever cell's output
looks best, or use the "Download as PNG" option in Plotly's own toolbar (top-right of
the chart) to export a static image straight from Databricks.

In [0]:
%pip install --upgrade plotly
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F
import plotly.express as px
import plotly.graph_objects as go

# Re-aggregate to a coarser grid for plotting -- 60,000+ points renders very slowly
# in an interactive Plotly map; ~10,000-15,000 is smooth and still shows the real lanes.
PLOT_GRID_DEGREES = 1.0

traffic = spark.table("workspace.global_shipping.traffic_density_grid")
plot_grid = (
    traffic
    .withColumn("grid_lat", F.floor(F.col("lat") / PLOT_GRID_DEGREES) * PLOT_GRID_DEGREES)
    .withColumn("grid_lon", F.floor(F.col("lon") / PLOT_GRID_DEGREES) * PLOT_GRID_DEGREES)
    .groupBy("grid_lat", "grid_lon")
    .agg(F.sum("traffic_density").alias("traffic_density"))
    .toPandas()
)
print(f"Plotting {len(plot_grid):,} cells")

chokepoints = spark.table("workspace.global_shipping.chokepoints_final_comparison").toPandas()

Plotting 33,544 cells


## Option A: density heatmap + chokepoint markers on a "normal map" basemap

`Densitymap` / `Scattermap` (no "box" in the name) are the newer, tokenless,
MapLibre-based versions. `map_style="carto-voyager"` is CARTO's free, no-account
basemap designed to look like an ordinary consumer map (real land/water colors, roads,
city labels) rather than the moody, minimal "darkmatter" style. Satellite imagery isn't
included here on purpose -- every real satellite basemap provider (Google, Mapbox,
Esri) requires a paid account beyond a small free quota, which is exactly what we're
avoiding.

In [0]:
fig = go.Figure()

fig.add_trace(go.Densitymap(
    lat=plot_grid["grid_lat"], lon=plot_grid["grid_lon"], z=plot_grid["traffic_density"],
    radius=10, colorscale="Teal", showscale=False, opacity=0.75,
))

fig.add_trace(go.Scattermap(
    lat=chokepoints["center_lat"], lon=chokepoints["center_lon"],
    mode="markers+text",
    marker=dict(
        size=8 + 26 * (chokepoints["capacity_share"] / chokepoints["capacity_share"].max()) ** 0.5,
        color="#A9782E", opacity=0.9,
    ),
    text=chokepoints["chokepoint"],
    textposition="top center",
    textfont=dict(color="black", size=12),
    hovertext=[
        f"{row.chokepoint}<br>Density share: {row.share_of_ranked_total:.1%}<br>Capacity share (IMF): {row.capacity_share:.1%}"
        for row in chokepoints.itertuples()
    ],
    hoverinfo="text",
))

fig.update_layout(
    map_style="carto-voyager",  # free, no token needed -- looks like an ordinary Google-Maps-style map
    map_zoom=1, map_center={"lat": 20, "lon": 20},
    margin=dict(l=0, r=0, t=40, b=0),
    height=650,
    title="Real AIS traffic density vs. real chokepoint capacity (brass markers, sized by IMF capacity share)",
    showlegend=False,
)
fig.show()

<!doctype html>

## Option B: same data, the clean "BI dashboard" basemap

`carto-positron` is a light, minimal, no-clutter style -- the same family of basemap
Tableau and Power BI tend to default to, if voyager's colors feel too busy for a
printed figure.

In [0]:
fig2 = go.Figure(fig)
fig2.update_layout(map_style="carto-positron")
fig2.show()

<!doctype html>

## Option C: just the 9 chokepoints, no density layer, paired-bar style hover

Useful if the density heatmap ends up feeling too busy for a clean static image.

In [0]:
fig3 = px.scatter_map(
    chokepoints, lat="center_lat", lon="center_lon",
    size="avg_daily_capacity", color="capacity_share",
    color_continuous_scale="Oranges",
    hover_name="chokepoint",
    hover_data={"share_of_ranked_total": ":.1%", "capacity_share": ":.1%", "center_lat": False, "center_lon": False},
    zoom=1, height=600,
    title="The 9 chokepoints, sized and colored by real cargo capacity share",
)
fig3.update_layout(map_style="carto-voyager", margin=dict(l=0, r=0, t=40, b=0))
fig3.show()

<!doctype html>